# 1. Data Preprocessing, Normalizing: Tiền xử lí dữ liệu, EDA, Chuẩn hóa dữ liệu.

In [ ]:
import json
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
from torch.utils.data import Dataset

# 1. Đọc file Dataset MỚI
with open('Gen dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

texts = []
labels = []
for intent in dataset['intents']:
    for pattern in intent['patterns']:
        texts.append(pattern)
        labels.append(intent['tag'])

# 2. Mã hóa Nhãn
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

# 3. CHIA DỮ LIỆU (80% Học - 20% Thi thử)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)

# 4. Tokenize bằng BERT
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# 5. Đóng gói cho PyTorch
class MentalHealthDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalHealthDataset(train_encodings, train_labels)
val_dataset = MentalHealthDataset(val_encodings, val_labels)

print(f"✅ Hoàn tất! Số class: {num_classes}")
print(f"📚 Dữ liệu học: {len(train_texts)} câu | 🎯 Dữ liệu thi thử: {len(val_texts)} câu")

# 2. Model Architecture: Xây dựng mô hình.

In [ ]:
from transformers import AutoModelForSequenceClassification

# Tải BERT bản gốc và thay cái "cửa ra" (num_labels) bằng số lượng tags của bạn
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased', 
    num_labels=num_classes
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"✅ Đã tải thành công lên: {device.type.upper()}")
# (Lưu ý: Bạn cứ phớt lờ cảnh báo màu đỏ "Some weights..." của Hugging Face nhé, đó không phải lỗi)

# 3. Model Training: Train Mô hình Chatbot.

In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import pickle

# Hàm tính toán chỉ số Accuracy và F1
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=8,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy="epoch",  # Cứ xong 1 epoch là tự động làm bài thi thử
    save_strategy="epoch",
    load_best_model_at_end=True,  # Tự động giữ lại phiên bản điểm cao nhất
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,     # Đưa tập thi thử vào
    compute_metrics=compute_metrics # Gọi hàm tính điểm
)

print("🚀 Start training Meiritsuu...")
trainer.train()

model.save_pretrained("./bert_chatbot_model")
tokenizer.save_pretrained("./bert_chatbot_model")
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
print("✅ Đã lưu xong mô hình xuất sắc nhất!")

# 4. Infering & Chatting: Tạo luồng - Chạy thử bot.

In [ ]:
import torch
import random
import json
import pickle
import requests # Dùng để gọi Ollama
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. TẢI "CÔ LỄ TÂN" BERT
model = AutoModelForSequenceClassification.from_pretrained("./bert_chatbot_model")
tokenizer = AutoTokenizer.from_pretrained("./bert_chatbot_model")
with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
with open('Gen dataset - Copy.json', 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

bot_name = "Meiritsuu"
print(f"🤖 {bot_name} (BERT + LLM) is ready!~ Press 'quit' to exit~~~.")
print("-" * 50)

# Hàm gửi lệnh cho "Bác sĩ" Llama-3 qua API của Ollama
def ask_llm(user_input, medical_context):
    # Prompt cho mô hình 3B cần ngắn gọn, đi thẳng vào vấn đề
    prompt = f"""You are Pandora, an empathetic mental health AI assistant. 
User says: "{user_input}"
Medical Knowledge to base your answer on: "{medical_context}"

Task: Write a short, caring, and conversational response to the user. 
Rule 1: Use ONLY the provided Medical Knowledge. Do NOT make up facts.
Rule 2: Keep it concise (under 3 sentences)."""

    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3.2",   # <--- CHỈ CẦN THAY ĐỔI TÊN MÔ HÌNH Ở DÒNG NÀY
        "prompt": prompt,
        "stream": False
    }
    response = requests.post(url, json=payload)
    return response.json()['response']

# 2. VÒNG LẶP CHAT
while True:
    sentence = input("Bạn: ")
    if sentence.lower() == "quit":
        break

    # --- BƯỚC 1: BERT PHÂN LUỒNG ---
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()
    probs = torch.softmax(logits, dim=1)
    prob = probs[0][predicted_class_id].item()

    if prob > 0.65:
        tag_name = label_encoder.inverse_transform([predicted_class_id])[0]
        
        # Công tắc an toàn
        if tag_name == "crisis_emergence":
            print(f"{bot_name}: TÔI RẤT LO LẮNG CHO BẠN. Hãy gọi ngay đường dây nóng 111 hoặc 115! Bạn không cô đơn.")
            continue
            
        # Tìm kiến thức y khoa gốc
        medical_context = ""
        for intent in intents_data['intents']:
            if tag_name == intent["tag"]: 
                medical_context = random.choice(intent['responses'])
                break
                
        # --- BƯỚC 2: LLM SINH VĂN BẢN TỰ NHIÊN ---
        print(f"*(BERT xác định tag: {tag_name} - Đang chờ Llama suy nghĩ...)*")
        final_answer = ask_llm(sentence, medical_context)
        print(f"{bot_name}: {final_answer}")
        
    else:
        print(f"{bot_name}: Xin lỗi, tôi không chắc mình hiểu rõ ý bạn (Tự tin: {prob:.2f}). Bạn có thể nói rõ hơn không?")